In [ ]:
!pip install -U chronos-forecasting -q

In [ ]:
import numpy as np
import torch
from google.colab import drive
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report
from chronos import Chronos2Pipeline

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', device)

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
)
print('Chronos-2 loaded')

def get_chronos2_embeddings(pipeline, X, batch_size=16):
    all_embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        context = torch.tensor(batch, dtype=torch.float32)
        embeddings, _ = pipeline.embed(context)
        for emb in embeddings:
            pooled = emb.mean(dim=1).flatten().numpy()
            all_embeddings.append(pooled)
        print(f'  {min(i+batch_size, len(X))}/{len(X)} done', flush=True)
    return np.array(all_embeddings)

In [ ]:
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

Z_tr = get_chronos2_embeddings(pipeline, X_tr)
Z_te = get_chronos2_embeddings(pipeline, X_te)

clf = RidgeClassifier()
clf.fit(Z_tr, y_tr)
y_pred = clf.predict(Z_te)

acc = accuracy_score(y_te, y_pred)
print(f'Chronos-2 Accuracy on CHBMIT: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_train_emb.npy', Z_tr)
np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_test_emb.npy', Z_te)

In [ ]:
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient.npy', allow_pickle=True).item()
batch = data['train']['X'][:4]
context = torch.tensor(batch, dtype=torch.float32)
embeddings, _ = pipeline.embed(context)
print('type:', type(embeddings))
print('len:', len(embeddings))
print('one element shape:', embeddings[0].shape)

In [ ]:
DESYNC_SEED = 0

def desync_channels(X, seed):
    r = np.random.default_rng(seed)
    out = X.copy()
    for w in range(out.shape[0]):
        for c in range(out.shape[1]):
            out[w, c] = np.roll(out[w, c], r.integers(0, out.shape[2]))
    return out

data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

X_tr_d = desync_channels(X_tr, DESYNC_SEED)
X_te_d = desync_channels(X_te, DESYNC_SEED)

Z_tr = get_chronos2_embeddings(pipeline, X_tr_d)
Z_te = get_chronos2_embeddings(pipeline, X_te_d)

clf = RidgeClassifier()
clf.fit(Z_tr, y_tr)
y_pred = clf.predict(Z_te)

acc = accuracy_score(y_te, y_pred)
print(f'Chronos-2 Accuracy on CHBMIT (channel-desynced): {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_desync_train_emb.npy', Z_tr)
np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_desync_test_emb.npy', Z_te)

In [ ]:
def get_chronos2_channelpooled(pipeline, X, batch_size=16):
    all_embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        context = torch.tensor(batch, dtype=torch.float32)
        embeddings, _ = pipeline.embed(context)
        for emb in embeddings:
            pooled = emb.mean(dim=(0, 1)).flatten().numpy()
            all_embeddings.append(pooled)
        print(f'  {min(i+batch_size, len(X))}/{len(X)} done', flush=True)
    return np.array(all_embeddings)

data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

Z_tr = get_chronos2_channelpooled(pipeline, X_tr)
Z_te = get_chronos2_channelpooled(pipeline, X_te)

clf = RidgeClassifier(alpha=10.0)
clf.fit(Z_tr, y_tr)
y_pred = clf.predict(Z_te)

acc = accuracy_score(y_te, y_pred)
print(f'Chronos-2 Accuracy on CHBMIT (channel-pooled): {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_chanpool_train_emb.npy', Z_tr)
np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_chanpool_test_emb.npy', Z_te)